# 환율 Gold 적재

입력된 일자 파라미터로 workspace.silver.exchange_rate_koreaexim 테이블에서 해당 일자의 데이터를 조회하여 workspace.gold.exchange_rate_koreaexim 테이블로 저장합니다.

In [0]:
# ── 일자 파라미터 입력 ─────────────────────────────
#dbutils.widgets.text("search_date", "20261012", "조회 일자 (YYYYMMDD)")
search_date = dbutils.widgets.get("search_date")
print(f"📋 조회 일자: {search_date}")

# ── Gold 스키마 생성 ────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

# ── Silver 테이블에서 해당 일자 데이터 조회 ───────────
silver_df = spark.read.table("workspace.silver.exchange_rate_koreaexim")
gold_df = silver_df.filter(silver_df.insert_date == search_date)

record_count = gold_df.count()
print(f"Silver → Gold 대상 건수: {record_count}건")

if record_count == 0:
    print("⚠️ 해당 일자의 데이터가 없습니다.")
else:
    # ── Gold 테이블에 데이터 저장 (해당 일자 overwrite) ──
    # 기존 Gold 테이블에서 해당 일자 데이터를 삭제 후 새로 저장
    try:
        spark.sql(f"DELETE FROM workspace.gold.exchange_rate_koreaexim WHERE insert_date = '{search_date}'")
    except Exception:
        pass  # 테이블이 없거나 해당 일자 데이터가 없는 경우 무시

    gold_df.write \
        .mode("append") \
        .saveAsTable("workspace.gold.exchange_rate_koreaexim")

    print(f"✅ Gold 테이블 저장 완료: workspace.gold.exchange_rate_koreaexim ({record_count}건)")

📋 조회 일자: 20261012
Silver → Gold 대상 건수: 96건
✅ Gold 테이블 저장 완료: workspace.gold.exchange_rate_koreaexim (96건)


In [0]:
# ── Gold 테이블 데이터 확인 ───────────────────────
gold_result_df = spark.read.table("workspace.gold.exchange_rate_koreaexim")
print(f"Gold 테이블 총 레코드 수: {gold_result_df.count()}")
print(f"컬럼: {gold_result_df.columns}")
display(gold_result_df.filter(gold_result_df.insert_date == search_date).limit(10))

Gold 테이블 총 레코드 수: 96
컬럼: ['result', 'cur_unit', 'cur_nm', 'ttb', 'tts', 'deal_bas_r', 'bkpr', 'yy_efee_r', 'ten_dd_efee_r', 'kftc_bkpr', 'kftc_deal_bas_r', 'insert_date']


result,cur_unit,cur_nm,ttb,tts,deal_bas_r,bkpr,yy_efee_r,ten_dd_efee_r,kftc_bkpr,kftc_deal_bas_r,insert_date
3,null,null,null,null,null,null,null,null,null,null,20261012
3,null,null,null,null,null,null,null,null,null,null,20261012
1,AED,아랍에미리트 디르함,360.65,367.94,364.3,364,0,0,364,364.3,20261012
1,AUD,호주 달러,925.38,944.07,934.73,934,0,0,934,934.73,20261012
1,BHD,바레인 디나르,"3,507.53","3,578.38","3,542.96","3,542",0,0,"3,542","3,542.96",20261012
1,BND,브루나이 달러,"1,034.21","1,055.1","1,044.66","1,044",0,0,"1,044","1,044.66",20261012
1,CAD,캐나다 달러,928.83,947.6,938.22,938,0,0,938,938.22,20261012
1,CHF,스위스 프랑,"1,594.87","1,627.08","1,610.98","1,610",0,0,"1,610","1,610.98",20261012
1,CNH,위안화,197.58,201.57,199.58,199,0,0,199,199.58,20261012
1,DKK,덴마아크 크로네,198.46,202.47,200.47,200,0,0,200,200.47,20261012
